# Wikipedia Stability and Contention - Full EN Data Collection (v4)
### IS-618 Social Media Data Analysis - University of Mannheim
### Marmee Pandya - EN Lead

---

## What this notebook does

This is the **full English data collection notebook** for the final study.
It builds on all pilot iterations (v1, v2, v3) with every fix applied.

Target: **300 contested + 300 stable EN articles** (matched to 300 pairs)

---

## Key decisions made across all pilot iterations

| Decision | Reason |
|---|---|
| Templates: POV + Neutrality only | Language-level disputes only - not sourcing or formatting |
| Date filter: 2022-2025 | Recent enough for active disputes, old enough for real conflict |
| Stable source: Good Articles | Community-reviewed, earned stability - not random stubs |
| Stable verification: no dispute history ever | Dual criterion makes stable label defensible |
| Min words: 1500 (both groups) | Same floor ensures comparability |
| Min editors: 15 (both groups) | Ensures article had enough attention to potentially be disputed |
| Max words stable: 8000 | Prevents Good Articles being 20K word mismatches |
| Topic model: Lift Wing outlink | Language-agnostic, works for EN and DE, same taxonomy |
| Matching: nearest-neighbour on 4 variables | Controls for age, length, topic, attention |
| Rate limit wait: 180s | Prevents repeated blocking |
| Save every 5 articles | Safe to interrupt anytime |
| Topic quotas: balanced across 4 buckets | Experiment 4 showed geography dominated at 49% - distorted results |

---

## What changed from v3

- Target raised from 50 to 300 per group
- Date filter explicitly 2022-2025 (drops 2026 - too recent)
- Topic quotas added - balanced collection across topic buckets
- spaCy stylometric features added (passive voice, named entity density)
- Wilcoxon signed-rank used instead of Mann-Whitney (correct for matched pairs)
- All files use v4_ prefix

---

## For the next person reading this

The pipeline works like this:

1. Run cells 0-3 (setup, config, helpers, fetch function) - defines everything
2. Run cell 4 (contested collection) - resumes if interrupted, stops at quota
3. Run cell 5 (stable collection) - resumes if interrupted
4. Run cell 6 (trends enrichment) - optional, skip if pytrends blocked
5. Run cell 7 (matching) - pairs contested with stable
6. Run cell 8 (feature extraction) - resumes from CSV
7. Run cells 9-12 (analysis) - stats, model, results

All JSON files save automatically. If interrupted anywhere just rerun that cell.
Files: v4_contested.json, v4_stable_pool.json, v4_matched_pairs.json, v4_features.csv


## 0. Setup and Imports

In [1]:
# Install if needed:
# !pip install requests pytrends lexicalrichness scikit-learn pandas numpy
# !pip install matplotlib seaborn spacy sentence-transformers
# !python -m spacy download en_core_web_sm

import requests, time, json, re, random, os, datetime
from datetime import timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from scipy.stats import wilcoxon, mannwhitneyu, norm

# spaCy for stylometric features
try:
    import spacy
    nlp_en = spacy.load("en_core_web_sm")
    SPACY_AVAILABLE = True
    print("spaCy loaded")
except:
    SPACY_AVAILABLE = False
    print("spaCy not available - stylometric features will be skipped")
    print("Run: python -m spacy download en_core_web_sm")

try:
    from pytrends.request import TrendReq
    TRENDS_AVAILABLE = True
    print("pytrends loaded")
except:
    TRENDS_AVAILABLE = False
    print("pytrends not available - trends matching skipped")

HEADERS = {"User-Agent": "UniMannheim-SMDA-v4/1.0 (student research uni-mannheim.de)"}
random.seed(42)
print("Ready")


spaCy loaded
pytrends loaded
Ready


## 1. Configuration

In [2]:
# File names - v4 prefix, v3 files untouched
V4_CONTESTED_FILE = "final_contested_en.json"
V4_STABLE_FILE    = "final_stable_en.json"
V4_PAIRS_FILE     = "final_pairs_en.json"
V4_FEATURES_FILE  = "final_features_en.csv"
V4_TRENDS_C       = "final_trends_contested.json"
V4_TRENDS_S       = "final_trends_stable.json"
V4_GOOD_TITLES    = "final_good_titles_en.json"

# Target - full study scale
TARGET_CONTESTED   = 200
TARGET_STABLE_POOL = 400   # collect 2x target to have matching headroom

# Quality thresholds - same for both groups
# Rationale: same bar ensures comparability
# Stability reflects genuine consensus not low visibility
MIN_WORDS    = 1500
MIN_EDITORS  = 15
MIN_AGE_DAYS = 365
MAX_WORDS_STABLE = 8000   # upper cap for stable only

# Matching constraints (validated across v1-v3)
MAX_MATCH_DISTANCE = 5.0
MAX_AGE_GAP_DAYS   = 730
WORD_COUNT_TOL     = 0.20

# Templates - language-level disputes only
# Validated: POV and Neutrality are the only EN templates
# with sufficient 2022-2025 coverage (52% pass rate confirmed)
EN_TEMPLATES = ["Template:POV", "Template:Neutrality"]

# Date filter - 2022 to 2025 only
# 2026 dropped: articles too recent, may not have had time
# to develop real editorial conflict (confirmed by professor)
VALID_YEARS = [2022, 2023, 2024, 2025]

# Lift Wing topic mapping
TOPIC_BROAD = {
    "STEM":                "science",
    "Culture":             "culture",
    "Geography":           "geography",
    "History_and_Society": "politics_history",
}

# Topic quotas - prevents geography domination
# Experiment 4 showed geography at 49% distorts results (F1 drop of 0.098)
# Target distribution based on pilot findings:
# politics_history shows strongest signal, science too rare naturally
TOPIC_QUOTAS_CONTESTED = {
    "politics_history": 50,
    "culture":          50,
    "geography":        50,
    "science":          50,
}
TOPIC_QUOTAS_STABLE = {
    "politics_history": 100,
    "culture":          100,
    "geography":        100,
    "science":          100,
}

def quota_reached(articles, topic, quotas):
    current = sum(1 for a in articles if a.get("topic") == topic)
    limit   = quotas.get(topic, 10)
    return current >= limit

def topic_summary(articles):
    from collections import Counter
    counts = Counter(a.get("topic","unknown") for a in articles)
    total  = max(len(articles), 1)
    for t, c in sorted(counts.items()):
        print(f"  {t:<20} {c:>4} ({100*c//total}%)")

print("Config loaded")
print(f"  Target contested   : {TARGET_CONTESTED}")
print(f"  Target stable pool : {TARGET_STABLE_POOL}")
print(f"  Min words          : {MIN_WORDS}")
print(f"  Min editors        : {MIN_EDITORS}")
print(f"  Date filter        : {VALID_YEARS}")
print(f"  Rate limit wait    : 180s")
print(f"  Topic quotas active: YES")


Config loaded
  Target contested   : 200
  Target stable pool : 400
  Min words          : 1500
  Min editors        : 15
  Date filter        : [2022, 2023, 2024, 2025]
  Rate limit wait    : 180s
  Topic quotas active: YES


## 2. Helper Functions

In [3]:
def safe_get(url, params, retries=4, base_wait=3):
    """Robust Wikipedia API GET with retry and rate limit handling."""
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=20)
            if r.status_code == 429:
                print("    Rate limited - waiting 180s (3 minutes)...")
                time.sleep(180)
                continue
            if r.status_code == 200 and r.text.strip():
                return r.json()
            time.sleep(base_wait * (attempt + 1))
        except Exception as e:
            print(f"    Error: {e}")
            time.sleep(base_wait * (attempt + 1))
    return None


def save_json(data, path):
    """Atomic save - temp file then replace. Partial writes never corrupt."""
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default=None):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return default if default is not None else []


def is_recent_dispute(text):
    """Check if POV or Neutrality template added in 2022-2025."""
    templates = re.findall(
        r'\{\{(?:POV|Neutrality)[^}]*\}\}', text, re.IGNORECASE)
    for t in templates:
        m = re.search(r'date=\w+ (20\d{2})', t)
        if m and int(m.group(1)) in VALID_YEARS:
            return True
    return False


def clean_wikitext(text):
    """Strip Wikipedia markup and return plain readable text."""
    text = re.sub(r'\{\{[^{}]*\}\}', '', text)
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]*)\]\]', r'\1', text)
    text = re.sub(r'==+[^=]+=+', '', text)
    text = re.sub(r'<ref[^>]*>.*?</ref>', '', text, flags=re.DOTALL)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r"'{2,}", '', text)
    return re.sub(r'\s+', ' ', text).strip()


def get_topic_liftwing(title, lang, retries=3):
    """
    Get topic using Lift Wing outlink model.
    Confirmed working for EN and DE with same taxonomy.
    Low confidence predictions (score < 0.5) return 'other'.
    """
    url = "https://api.wikimedia.org/service/lw/inference/v1/models/outlink-topic-model:predict"
    for attempt in range(retries):
        try:
            r = requests.post(url,
                headers={**HEADERS, "Content-Type": "application/json"},
                json={"page_title": title, "lang": lang},
                timeout=15)
            if r.status_code == 200:
                results = r.json()["prediction"]["results"]
                if not results:
                    return "other", None
                top = max(results, key=lambda x: x["score"])
                if top["score"] < 0.5:
                    return "other", top["topic"]
                specific = top["topic"]
                prefix   = specific.split(".")[0]
                broad    = TOPIC_BROAD.get(prefix, "other")
                return broad, specific
            time.sleep(2 * (attempt + 1))
        except:
            time.sleep(2 * (attempt + 1))
    return "other", None


def get_trends(title, cache):
    """Fetch Google Trends score with caching."""
    if not TRENDS_AVAILABLE or title in cache:
        return cache.get(title)
    try:
        pt = TrendReq(hl='en-US', tz=360)
        pt.build_payload([title], timeframe='today 12-m', geo='')
        df_t = pt.interest_over_time()
        val = round(df_t[title].mean(), 1) if not df_t.empty and title in df_t else None
        cache[title] = val
        return val
    except:
        cache[title] = None
        return None

print("Helper functions ready")


Helper functions ready


## 3. Article Fetch Function

In [4]:
def fetch_article(title, label, lang="en"):
    """
    Fetch full article data from Wikipedia API.

    label=0 = contested (applies date filter, checks for POV/Neutrality template)
    label=1 = stable (applies max word cap, no dispute template required)

    Same MIN_WORDS and MIN_EDITORS for both groups.
    Returns dict or None if article fails any filter.

    API calls per article:
    1. Main text + categories (for content and topic)
    2. Edit history (for editor count, age, recency)
    3. Talk page (for behavioral signals)
    4. Lift Wing (for topic classification)
    Total: ~4 API calls + 1 external call per article
    """
    url = f"https://{lang}.wikipedia.org/w/api.php"

    # Call 1: Text and categories
    d1 = safe_get(url, {
        "action": "query", "titles": title,
        "prop": "revisions|categories",
        "rvprop": "content|timestamp", "rvslots": "main",
        "rvlimit": 1, "cllimit": 50, "format": "json"
    })
    if not d1: return None
    page = list(d1["query"]["pages"].values())[0]
    if "revisions" not in page: return None

    raw = page["revisions"][0]["slots"]["main"]["*"]
    wc  = len(raw.split())

    if wc < MIN_WORDS: return None
    if label == 1 and wc > MAX_WORDS_STABLE: return None
    if label == 0 and not is_recent_dispute(raw): return None

    clean     = clean_wikitext(raw)
    citations = len(re.findall(r'<ref', raw, re.IGNORECASE))
    sections  = len(re.findall(r'^==+[^=]', raw, re.MULTILINE))

    # Lift Wing topic (1 external call)
    broad_topic, specific_topic = get_topic_liftwing(title, lang)
    time.sleep(1.0)

    time.sleep(1.2)

    # Call 2: Edit history
    d2 = safe_get(url, {
        "action": "query", "titles": title,
        "prop": "revisions", "rvprop": "user|timestamp",
        "rvlimit": 500, "format": "json"
    })
    revs = []
    if d2:
        revs = list(d2["query"]["pages"].values())[0].get("revisions", [])

    unique_eds  = len(set(r.get("user", "") for r in revs))
    total_edits = len(revs)
    if unique_eds < MIN_EDITORS: return None

    first = revs[-1]["timestamp"] if revs else None
    age_days = 0
    if first:
        age_days = (datetime.datetime.now(timezone.utc) -
                    datetime.datetime.fromisoformat(
                        first.replace("Z", "+00:00"))).days
        if age_days < MIN_AGE_DAYS: return None

    cutoff = datetime.datetime(2025, 10, 1, tzinfo=timezone.utc)
    recent = sum(1 for r in revs
                 if datetime.datetime.fromisoformat(
                     r["timestamp"].replace("Z", "+00:00")) >= cutoff)
    recency = round(recent / total_edits, 3) if total_edits else 0

    time.sleep(1.2)

    # Call 3: Talk page
    d3 = safe_get(url, {
        "action": "query", "titles": f"Talk:{title}",
        "prop": "revisions", "rvprop": "comment|content|user",
        "rvslots": "main", "rvlimit": 500, "format": "json"
    })
    revert_count = talk_words = talk_editors = 0
    if d3:
        tp    = list(d3["query"]["pages"].values())[0]
        trevs = tp.get("revisions", [])
        revert_count = sum(1 for r in trevs
                           if any(w in r.get("comment", "").lower()
                                  for w in ["revert", "reverted", "undid",
                                            "undo", "restored"]))
        talk_editors = len(set(r.get("user", "") for r in trevs))
        if trevs and "slots" in trevs[0]:
            talk_words = len(
                trevs[0]["slots"]["main"].get("*", "").split())

    return {
        "title":          title,
        "label":          label,
        "label_name":     "contested" if label == 0 else "stable",
        "raw_text":       raw,
        "clean_text":     clean,
        "word_count":     wc,
        "topic":          broad_topic,
        "topic_specific": specific_topic,
        "citation_count": citations,
        "section_count":  sections,
        "unique_editors": unique_eds,
        "total_edits":    total_edits,
        "age_days":       age_days,
        "recency_ratio":  recency,
        "revert_count":   revert_count,
        "talk_words":     talk_words,
        "talk_editors":   talk_editors,
        "lang":           lang,
        "collected_at":   datetime.datetime.now().isoformat(),
    }

print("fetch_article ready")


fetch_article ready


## 4. Collect Contested Articles
Target: 300 articles with balanced topic distribution.
Saves every 5 articles. Resumes automatically if interrupted.
Cache expires after 7 days to catch newly tagged articles.
Topic quotas prevent geography dominance (confirmed distortion in Experiment 4).


In [5]:
def fetch_template_titles(template, limit=500):
    """Fetch titles for a dispute template. Cache expires after 7 days."""
    safe_name  = template.replace(':', '_').replace('/', '_')
    cache_file = f"v4_template_titles_{safe_name}.json"

    if os.path.exists(cache_file):
        age = (datetime.datetime.now() -
               datetime.datetime.fromtimestamp(
                   os.path.getmtime(cache_file))).days
        if age < 7:
            with open(cache_file, encoding="utf-8") as f:
                titles = json.load(f)
            print(f"  Loaded {len(titles)} from cache ({age}d old)")
            return titles
        print(f"  Cache {age}d old - re-fetching...")

    url = "https://en.wikipedia.org/w/api.php"
    titles, params = [], {
        "action": "query", "list": "embeddedin",
        "eititle": template, "eilimit": 500,
        "einamespace": 0, "format": "json"
    }
    while len(titles) < limit:
        data = safe_get(url, params)
        if not data: break
        titles.extend(p["title"] for p in data["query"]["embeddedin"])
        if "continue" not in data: break
        params["eicontinue"] = data["continue"]["eicontinue"]
        time.sleep(1)

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(titles, f, ensure_ascii=False)
    print(f"  Fetched and saved {len(titles)} titles")
    return titles


# Load existing progress
contested = load_json(V4_CONTESTED_FILE, [])
seen_c    = set(a["title"] for a in contested)
have      = len(contested)
need      = TARGET_CONTESTED - have

print(f"Contested: have {have}, need {need} more")
print("Current topic distribution:")
topic_summary(contested)

if need > 0:
    all_titles = []
    for t in EN_TEMPLATES:
        titles = fetch_template_titles(t, limit=2000)
        all_titles.extend(titles)
        print(f"  {t}: {len(titles)} titles")

    random.shuffle(all_titles)
    all_titles = [t for t in all_titles if t not in seen_c]
    print(f"  {len(all_titles)} unique candidates")

    skipped_quota = 0
    for title in all_titles:
        if len(contested) >= TARGET_CONTESTED: break
        time.sleep(random.uniform(5.0, 8.0))

        art = fetch_article(title, label=0, lang="en")

        if not art:
            print(f"  Skipped: {title[:45]}")
            continue

        if art["topic"] == "other":
            continue

        if quota_reached(contested, art["topic"], TOPIC_QUOTAS_CONTESTED):
            skipped_quota += 1
            if skipped_quota % 20 == 0:
                print(f"  ({skipped_quota} skipped for quota so far)")
            continue

    contested.append(art)
    seen_c.add(title)
    print(f"  [{len(contested)}/{TARGET_CONTESTED}] {title[:45]} "
          f"({art['word_count']}w, {art['topic']})")
    if len(contested) % 5 == 0:
        save_json(contested, V4_CONTESTED_FILE)
        print(f"  Saved at {len(contested)}")



print(f"\nContested total: {len(contested)}/{TARGET_CONTESTED}")
print("Final topic distribution:")
topic_summary(contested)


Contested: have 0, need 200 more
Current topic distribution:
  Fetched and saved 2000 titles
  Template:POV: 2000 titles
  Fetched and saved 49 titles
  Template:Neutrality: 49 titles
  2049 unique candidates
  Skipped: National Association of Corporate Directors
  Skipped: Gender-critical feminism


KeyboardInterrupt: 

## 5. Collect Stable Articles
Target: 600 pool (will be matched down to 300 pairs).
Good Articles verified to have no dispute template ever in revision history.
Same quality thresholds as contested (MIN_WORDS=1500, MIN_EDITORS=15).
Topic quotas mirror contested distribution.


In [ ]:
def has_dispute_history(title, lang="en"):
    """Check last 50 revisions for any dispute template."""
    url = f"https://{lang}.wikipedia.org/w/api.php"
    pat = re.compile(
        r'\{\{(?:POV|Disputed|Neutrality|Biased|Unbalanced)',
        re.IGNORECASE)
    data = safe_get(url, {
        "action": "query", "titles": title,
        "prop": "revisions", "rvprop": "content",
        "rvslots": "main", "rvlimit": 50, "format": "json"
    })
    if not data: return False
    page = list(data["query"]["pages"].values())[0]
    for rv in page.get("revisions", []):
        if pat.search(rv.get("slots", {}).get("main", {}).get("*", "")):
            return True
    return False


def fetch_good_article_titles():
    """
    Fetch Good Article titles.
    Loads from saved file if available (no re-fetching).
    Saves immediately after fetching so never lost again.
    """
    if os.path.exists(V4_GOOD_TITLES):
        with open(V4_GOOD_TITLES, encoding="utf-8") as f:
            titles = json.load(f)
        random.shuffle(titles)
        print(f"  Loaded {len(titles)} titles from file (shuffled)")
        return titles

    url    = "https://en.wikipedia.org/w/api.php"
    titles = []
    params = {
        "action": "query", "list": "categorymembers",
        "cmtitle": "Category:Good articles",
        "cmlimit": 500, "cmtype": "page",
        "cmnamespace": 0, "format": "json"
    }
    print("  Fetching ALL Good Article titles...")
    while True:
        data = safe_get(url, params)
        if not data: break
        batch = [p["title"] for p in data["query"]["categorymembers"]
                 if not p["title"].startswith("Talk:")]
        titles.extend(batch)
        print(f"  ... {len(titles)} fetched")
        if "continue" not in data: break
        params["cmcontinue"] = data["continue"]["cmcontinue"]
        time.sleep(1)

    random.shuffle(titles)
    with open(V4_GOOD_TITLES, "w", encoding="utf-8") as f:
        json.dump(titles, f, ensure_ascii=False)
    print(f"  Saved {len(titles)} titles to {V4_GOOD_TITLES}")
    return titles


# Load existing progress
stable = load_json(V4_STABLE_FILE, [])
seen_s = set(a["title"] for a in stable)
have_s = len(stable)
need_s = TARGET_STABLE_POOL - have_s

print(f"Stable pool: have {have_s}, need {need_s} more")
print("Current topic distribution:")
topic_summary(stable)

if need_s > 0:
    candidates = fetch_good_article_titles()
    candidates = [t for t in candidates if t not in seen_s]
    print(f"  {len(candidates)} new candidates\n")

    skipped_quota = 0
    for title in candidates:
        if len(stable) >= TARGET_STABLE_POOL: break
        time.sleep(random.uniform(5.0, 8.0))

        if has_dispute_history(title):
            print(f"  Skipped: {title[:45]} - dispute in history")
            continue

        art = fetch_article(title, label=1, lang="en")

        if not art:
            print(f"  Skipped: {title[:45]}")
            continue

        if art["topic"] == "other":
            continue

        if quota_reached(stable, art["topic"], TOPIC_QUOTAS_STABLE):
            skipped_quota += 1
            if skipped_quota % 20 == 0:
                print(f"  ({skipped_quota} skipped for quota so far)")
            continue

    stable.append(art)
    seen_s.add(title)
    print(f"  [{len(stable)}/{TARGET_STABLE_POOL}] {title[:45]} "
          f"({art['word_count']}w, {art['topic']})")
    if len(stable) % 5 == 0:
        save_json(stable, V4_STABLE_FILE)
        print(f"  Saved at {len(stable)}")

print(f"\nStable pool total: {len(stable)}/{TARGET_STABLE_POOL}")
print("Final topic distribution:")
topic_summary(stable)


## 6. Google Trends Enrichment (Optional)
Skip if pytrends is blocked. Matching falls back to age and word count.

In [ ]:
trends_c = load_json(V4_TRENDS_C, {})
trends_s = load_json(V4_TRENDS_S, {})

def enrich_trends(articles, cache, cache_file, label):
    missing = [a for a in articles if a["title"] not in cache]
    print(f"  Fetching trends for {len(missing)} {label} articles...")
    for i, art in enumerate(missing):
        get_trends(art["title"], cache)
        if (i + 1) % 10 == 0:
            save_json(cache, cache_file)
            print(f"  Saved at {i+1}/{len(missing)}")
        time.sleep(random.uniform(3, 5))
    save_json(cache, cache_file)
    for art in articles:
        art["trends_avg"] = cache.get(art["title"])
    return articles

if TRENDS_AVAILABLE:
    print("Enriching contested...")
    contested = enrich_trends(contested, trends_c, V4_TRENDS_C, "contested")
    print("Enriching stable...")
    stable    = enrich_trends(stable, trends_s, V4_TRENDS_S, "stable")
    print("Done")
else:
    for a in contested + stable:
        a.setdefault("trends_avg", None)
    print("Skipped - pytrends not available")
    print("Matching will use age and word count only")


## 7. Matching - Nearest Neighbour
Matches each contested article to the closest stable article on:
- Word count (within 20%)
- Article age (within 730 days)
- Google Trends average (if available)
- Topic preference: same broad topic preferred

Constraints validated across v1-v3 pilots.
Each stable article can only be matched once.
Pairs exceeding distance cutoff are rejected rather than forced.


In [ ]:
def match_articles(contested, stable_pool):
    used     = set()
    pairs    = []
    rejected = []

    def feats(a):
        return np.array([
            a.get("word_count", 0) / 10000,
            a.get("age_days",   0) / 3650,
            (a.get("trends_avg") or 0) / 100,
        ])

    stable_list  = list(stable_pool)
    stable_feats = np.array([feats(a) for a in stable_list])
    nn = NearestNeighbors(
        n_neighbors=min(30, len(stable_list)), metric="euclidean")
    nn.fit(stable_feats)

    for c in contested:
        dists, idxs = nn.kneighbors(feats(c).reshape(1, -1))
        matched = None

        for dist, idx in zip(dists[0], idxs[0]):
            if dist > MAX_MATCH_DISTANCE: break
            if idx in used: continue
            s = stable_list[idx]
            if abs(c["age_days"] - s["age_days"]) > MAX_AGE_GAP_DAYS:
                continue
            wc_ratio = (abs(c["word_count"] - s["word_count"]) /
                        max(c["word_count"], 1))
            if wc_ratio > WORD_COUNT_TOL: continue
            matched = (s, idx, dist)
            if s["topic"] == c["topic"]: break

        if matched:
            s, idx, dist = matched
            used.add(idx)
            pairs.append({
                "contested":      c,
                "stable":         s,
                "match_distance": round(dist, 4),
                "same_topic":     c["topic"] == s["topic"],
                "age_gap":        abs(c["age_days"] - s["age_days"]),
                "wc_gap_pct":     round(
                    abs(c["word_count"] - s["word_count"]) /
                    max(c["word_count"], 1) * 100, 1),
            })
        else:
            rejected.append(c["title"])

    return pairs, rejected


print("Running matching...")
pairs, rejected = match_articles(contested, stable)

same_topic  = sum(1 for p in pairs if p["same_topic"])
avg_dist    = np.mean([p["match_distance"] for p in pairs]) if pairs else 0
avg_age_gap = np.mean([p["age_gap"]        for p in pairs]) if pairs else 0
avg_wc_gap  = np.mean([p["wc_gap_pct"]     for p in pairs]) if pairs else 0

print(f"\nMatched  : {len(pairs)} pairs")
print(f"Rejected : {len(rejected)} contested articles (no valid match)")
print(f"Same topic    : {same_topic}/{len(pairs)}")
print(f"Avg distance  : {avg_dist:.4f}")
print(f"Avg age gap   : {avg_age_gap:.0f} days")
print(f"Avg word gap  : {avg_wc_gap:.1f}%")

if rejected:
    print("\nRejected (may need more stable articles in these topics):")
    for t in rejected[:10]:
        print(f"  - {t}")

save_json([{
    "contested":      p["contested"],
    "stable":         p["stable"],
    "match_distance": p["match_distance"],
    "same_topic":     p["same_topic"],
    "age_gap":        p["age_gap"],
    "wc_gap_pct":     p["wc_gap_pct"],
} for p in pairs], V4_PAIRS_FILE)
print(f"\nSaved to {V4_PAIRS_FILE}")


## 8. Feature Extraction
Features validated across pilot iterations v1-v3.

Key findings from pilot experiments:
- Structural features strongest (F1=0.704 alone)
- Behavioral (talk page) features add signal (F1=0.593 alone)
- Linguistic features weak at n=40 but directionally correct (F1=0.538)
- Expanded hedging list DILUTED signal - keep original 20 words only
- Wilcoxon signed-rank more appropriate than Mann-Whitney for matched pairs

New in v4: spaCy stylometric features (passive voice, named entity density)
These are theoretically motivated - contested articles may use passive voice
to avoid attributing disputed claims to specific actors.


In [ ]:
# Hedging word list - validated in pilot
# DO NOT expand - Experiment 2 showed expansion dilutes signal
# These 20 words are the core epistemic hedging markers
HEDGING_WORDS = [
    "allegedly", "apparently", "arguably", "claims", "claimed",
    "reportedly", "supposedly", "some argue", "some suggest",
    "it is claimed", "it has been argued", "according to some",
    "disputed", "controversial", "contentious", "debated",
    "critics say", "critics argue", "opponents claim",
    "proponents argue", "others believe", "many believe",
]

DEFINITION_PATTERNS = [
    r'\bis (a|an|the)\b', r'\brefers to\b', r'\bdefined as\b',
    r'\bknown as\b', r'\bdescribed as\b', r'\bconsidered (a|an|the)\b',
]


def get_stylometric_features(text):
    """
    spaCy-based stylometric features.
    Requires en_core_web_sm to be installed.
    Returns dict of features or None if spaCy unavailable.

    Passive voice ratio: contested articles may avoid naming actors
    in disputed claims by using passive constructions.

    Named entity density: contested articles about specific people
    and events may have higher entity density as disputes often
    centre on factual claims about specific named subjects.
    """
    if not SPACY_AVAILABLE:
        return {"passive_ratio": None, "ent_density": None,
                "avg_sent_len": None}
    try:
        doc       = nlp_en(text[:50000])
        sents     = list(doc.sents)
        n_sents   = max(len(sents), 1)
        n_tokens  = max(len(doc), 1)
        passive   = sum(1 for t in doc if t.dep_ == "nsubjpass")
        passive_r = round(passive / n_sents, 4)
        ent_d     = round(len(doc.ents) / n_tokens * 1000, 4)
        avg_sl    = round(
            sum(len(list(s)) for s in sents) / n_sents, 2)
        return {
            "passive_ratio": passive_r,
            "ent_density":   ent_d,
            "avg_sent_len":  avg_sl,
        }
    except:
        return {"passive_ratio": None, "ent_density": None,
                "avg_sent_len": None}


def extract_features(article):
    """Extract all features from one article."""
    text  = article.get("clean_text", "")
    words = text.lower().split()
    n     = len(words)
    if n == 0: return None

    sents  = [s.strip() for s in re.split(r'[.!?]+', text)
               if len(s.strip()) > 10]
    n_sent = max(len(sents), 1)
    tlow   = text.lower()

    hedge   = sum(tlow.count(w) for w in HEDGING_WORDS)
    hedging = round(hedge / n * 1000, 4)

    def_s = sum(1 for s in sents
                if any(re.search(p, s, re.IGNORECASE)
                       for p in DEFINITION_PATTERNS))
    def_r = round(def_s / n_sent, 4)

    stylo = get_stylometric_features(text)

    return {
        "title":           article["title"],
        "label":           article["label"],
        "label_name":      article["label_name"],
        "topic":           article["topic"],
        "topic_specific":  article.get("topic_specific"),
        "lang":            article.get("lang", "en"),
        # Linguistic
        "hedging_density": hedging,
        "def_ratio":       def_r,
        # Stylometric (new in v4)
        "passive_ratio":   stylo["passive_ratio"],
        "ent_density":     stylo["ent_density"],
        "avg_sent_len":    stylo["avg_sent_len"],
        # Structural
        "citation_count":  article.get("citation_count", 0),
        "section_count":   article.get("section_count", 0),
        "unique_editors":  article.get("unique_editors", 0),
        "age_days":        article.get("age_days", 0),
        "recency_ratio":   article.get("recency_ratio", 0),
        # Behavioral (talk page)
        "revert_count":    article.get("revert_count", 0),
        "talk_words":      article.get("talk_words", 0),
        "talk_editors":    article.get("talk_editors", 0),
        # Matching reference (not in model)
        "word_count":      article.get("word_count", n),
        "trends_avg":      article.get("trends_avg"),
    }


# Resume from CSV if it exists
if os.path.exists(V4_FEATURES_FILE):
    df_existing  = pd.read_csv(V4_FEATURES_FILE)
    already_done = set(df_existing["title"].tolist())
    print(f"Loaded {len(df_existing)} existing rows from {V4_FEATURES_FILE}")
else:
    df_existing  = pd.DataFrame()
    already_done = set()
    print("No existing features file - extracting from scratch")

new_records = []
total = len(pairs) * 2
done  = 0
for p in pairs:
    for article in [p["contested"], p["stable"]]:
        if article["title"] in already_done:
            done += 1
            continue
        feat = extract_features(article)
        if feat:
            new_records.append(feat)
        done += 1
        if done % 20 == 0:
            print(f"  {done}/{total} articles processed")

if new_records:
    df_new = pd.DataFrame(new_records)
    df = pd.concat([df_existing, df_new], ignore_index=True)
else:
    df = df_existing

df.to_csv(V4_FEATURES_FILE, index=False)
print(f"\nFeature matrix: {df.shape[0]} articles x {df.shape[1]} columns")
print(f"  Contested : {(df.label==0).sum()}")
print(f"  Stable    : {(df.label==1).sum()}")
print(f"  spaCy features available: {df['passive_ratio'].notna().sum()}")
df.head(3)


## 9. Statistical Analysis
Wilcoxon signed-rank used (correct for matched-pair design).
Mann-Whitney also reported for comparison.

In [ ]:
c_df = df[df.label == 0]
s_df = df[df.label == 1]

# Base feature set
BASE_FEATURES = [
    "hedging_density", "def_ratio",
    "citation_count", "section_count",
    "unique_editors", "age_days", "recency_ratio",
    "revert_count", "talk_words", "talk_editors",
]

# Add stylometric if available
STYLO_FEATURES = []
if df["passive_ratio"].notna().sum() > 10:
    STYLO_FEATURES = ["passive_ratio", "ent_density", "avg_sent_len"]
    print("spaCy stylometric features included in analysis")

ALL_FEATURES = BASE_FEATURES + STYLO_FEATURES

LABELS = {
    "hedging_density": "Hedging Density",
    "def_ratio":       "Definition Ratio",
    "passive_ratio":   "Passive Voice Ratio",
    "ent_density":     "Named Entity Density",
    "avg_sent_len":    "Avg Sentence Length",
    "citation_count":  "Citation Count",
    "section_count":   "Section Count",
    "unique_editors":  "Unique Editors",
    "age_days":        "Article Age (days)",
    "recency_ratio":   "Edit Recency Ratio",
    "revert_count":    "Revert Count",
    "talk_words":      "Talk Word Count",
    "talk_editors":    "Talk Editors",
}

print(f"\n{'Feature':<25} {'Contested':>12} {'Stable':>12} {'Diff%':>8}")
print("-" * 62)
for col in ALL_FEATURES:
    if col not in df.columns: continue
    cm = c_df[col].mean()
    sm = s_df[col].mean()
    diff = (cm - sm) / (sm + 1e-9) * 100
    label = LABELS.get(col, col)
    print(f"{label:<25} {cm:>12.3f} {sm:>12.3f} "
          f"{'up' if diff>0 else 'dn'}{abs(diff):>6.1f}%")


In [ ]:
# Wilcoxon signed-rank (primary - correct for matched pairs)
print("Wilcoxon Signed-Rank Tests (primary - matched pair design)\n")
print(f"{'Feature':<25} {'Mean diff':>10} {'C>S %':>8} {'p-value':>10}  Sig?")
print("-" * 62)

# Build diff dataframe from pairs
diff_records = []
for p in pairs:
    c_title = p["contested"]["title"]
    s_title = p["stable"]["title"]
    c_row = df[df.title == c_title]
    s_row = df[df.title == s_title]
    if c_row.empty or s_row.empty: continue
    c_row = c_row.iloc[0]
    s_row = s_row.iloc[0]
    record = {}
    for feat in ALL_FEATURES:
        if feat in df.columns:
            record[f"diff_{feat}"] = c_row.get(feat, np.nan) - s_row.get(feat, np.nan)
    diff_records.append(record)

diff_df = pd.DataFrame(diff_records)
wilcox_results = []

for feat in ALL_FEATURES:
    col  = f"diff_{feat}"
    if col not in diff_df.columns: continue
    vals = diff_df[col].dropna()
    if len(vals) < 5: continue
    mean = vals.mean()
    pct  = (vals > 0).mean() * 100
    try:
        _, p = wilcoxon(vals)
        sig  = "YES" if p < 0.05 else ("~" if p < 0.10 else "no")
        label = LABELS.get(feat, feat)
        print(f"{label:<25} {mean:>10.3f} {pct:>7.1f}% {p:>10.4f}  {sig}")
        wilcox_results.append({"feature": feat, "p": p,
                                "mean_diff": mean, "pct_c_higher": pct,
                                "sig": p < 0.05})
    except:
        pass

print("\nYES = p<0.05   ~ = p<0.10   no = not significant")
print("Note: Wilcoxon signed-rank is more powerful than Mann-Whitney")
print("for matched-pair designs. Use these results as primary evidence.")


## 10. Logistic Regression + Topic Control

In [ ]:
# Add topic dummies as control variables
# Experiment 4 showed topic control reduces F1 by 0.098 - topic matters
df_topic = pd.get_dummies(df["topic"], prefix="topic", drop_first=True)
df_model = pd.concat([df.reset_index(drop=True),
                       df_topic.reset_index(drop=True)], axis=1)
topic_cols = list(df_topic.columns)

feature_sets = {
    "Linguistic only":     [f for f in ALL_FEATURES
                             if f in ["hedging_density","def_ratio",
                                      "passive_ratio","ent_density",
                                      "avg_sent_len"]],
    "Structural only":     ["citation_count","section_count",
                             "unique_editors","age_days","recency_ratio"],
    "Behavioral only":     ["revert_count","talk_words","talk_editors"],
    "All features":        ALL_FEATURES,
    "All + topic control": ALL_FEATURES + topic_cols,
}

cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
m   = LogisticRegression(class_weight="balanced",
                          max_iter=1000, random_state=42)

results_model = {}
print(f"{'Feature set':<25} {'F1':>8} {'Accuracy':>10} {'vs baseline':>12}")
print("-" * 58)

for name, feats in feature_sets.items():
    avail = [f for f in feats if f in df_model.columns]
    if len(avail) < 2:
        print(f"{name:<25} {'not enough features':>30}")
        continue
    d  = df_model[avail + ["label"]].dropna()
    X  = StandardScaler().fit_transform(d[avail].values)
    y  = d["label"].values
    if len(set(y)) < 2: continue
    f1s  = cross_val_score(m, X, y, cv=cv, scoring="f1_macro")
    accs = cross_val_score(m, X, y, cv=cv, scoring="accuracy")
    base = max(y.mean(), 1-y.mean())
    results_model[name] = f1s.mean()
    print(f"{name:<25} {f1s.mean():>8.3f} {accs.mean():>10.3f} "
          f"{accs.mean()-base:>+12.3f}")

# Coefficient chart for all features model
best_feats = [f for f in ALL_FEATURES if f in df_model.columns]
d_full = df_model[best_feats + ["label"]].dropna()
X_full = StandardScaler().fit_transform(d_full[best_feats].values)
y_full = d_full["label"].values
m.fit(X_full, y_full)

coef_df = pd.DataFrame({
    "feature":     [LABELS.get(f, f) for f in best_feats],
    "coefficient": m.coef_[0],
}).sort_values("coefficient")

fig, ax = plt.subplots(figsize=(9, max(6, len(best_feats)*0.5)))
colors  = ["#DC2626" if c < 0 else "#059669" for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"], coef_df["coefficient"],
        color=colors, alpha=0.85, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Standardised Coefficient (Contested <-- | --> Stable)",
               fontsize=11)
ax.set_title("v4 Logistic Regression Coefficients\n"
             "What predicts stability vs contestation?",
             fontsize=12, fontweight="bold")
ax.legend(handles=[
    mpatches.Patch(color="#DC2626", alpha=0.85, label="Predicts Contested"),
    mpatches.Patch(color="#059669", alpha=0.85, label="Predicts Stable"),
], loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("v4_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()


## 11. Study Summary

In [ ]:
print("=" * 70)
print("v4 ENGLISH DATA COLLECTION - SUMMARY")
print("=" * 70)

print(f"\nDataset")
print(f"  Contested articles : {len(contested)}/{TARGET_CONTESTED}")
print(f"  Stable pool        : {len(stable)}/{TARGET_STABLE_POOL}")
print(f"  Matched pairs      : {len(pairs)}")
print(f"  Rejected (no match): {len(rejected)}")
print(f"  Same-topic pairs   : {same_topic}/{len(pairs)}")

print(f"\nContested topic distribution:")
topic_summary(contested)
print(f"\nStable topic distribution:")
topic_summary(stable)

print(f"\nMatching Quality")
print(f"  Avg distance  : {avg_dist:.4f}")
print(f"  Avg age gap   : {avg_age_gap:.0f} days")
print(f"  Avg word gap  : {avg_wc_gap:.1f}%")

print(f"\nModel Performance (5-fold CV)")
for name, f1 in results_model.items():
    print(f"  {name:<25} F1={f1:.3f}")

sig_wilcox = [r for r in wilcox_results if r["sig"]]
print(f"\nSignificant features (Wilcoxon, p<0.05): {len(sig_wilcox)}")
for r in sorted(sig_wilcox, key=lambda x: x["p"]):
    label = LABELS.get(r["feature"], r["feature"])
    pct   = r["pct_c_higher"]
    print(f"  {label:<28} p={r['p']:.4f}  "
          f"C>S in {pct:.0f}% of pairs")

print(f"\nFiles")
print(f"  {V4_CONTESTED_FILE}")
print(f"  {V4_STABLE_FILE}")
print(f"  {V4_PAIRS_FILE}")
print(f"  {V4_FEATURES_FILE}")
print(f"  v4_coefficients.png")

print(f"\nHandoff notes for next person (Person 4 - Analysis)")
print("  1. Use Wilcoxon results as primary statistical evidence")
print("     (paired test, more appropriate than Mann-Whitney)")
print("  2. Include topic dummies as control variables in regression")
print("     (Experiment 4 showed 0.098 F1 drop without controls)")
print("  3. Replicate pipeline for German (Selma - Person 3)")
print("     Use de_core_news_sm for spaCy, Vorlage:Neutralitat template")
print("  4. Cross-lingual comparison: compare coefficients EN vs DE")
print("     Either convergence or divergence is a publishable finding")
print("=" * 70)


---
## Files and its content 

| File | Contents |
|---|---|
| v4_contested.json | 300 contested EN articles with full text and features |
| v4_stable_pool.json | 600 stable EN articles (Good Articles, no dispute history) |
| v4_matched_pairs.json | 300 matched pairs with distance and gap metadata |
| v4_features.csv | Full feature matrix ready for analysis |
| v4_good_article_titles_en.json | Cached Good Article titles (43K+) |
| v4_template_titles_*.json | Cached template title lists (7-day expiry) |
| v4_trends_*.json | Google Trends cache |
| v4_coefficients.png | Logistic regression coefficient chart |